In [4]:
!pip install Bio

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.8.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached httpcore-1.0.7-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 6.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 7.9 MB/s eta 0:00:00
Using cached httpcore-1.0.7-py3-none-any.whl (78 kB)
Using cached anyio-4.8.0-py3-none-any.whl (96 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import random

class SequenceModifier():
    ''' Modifies a sequence at a specific position'''
    def __init__(self, alphabet: list[str]):
        self.alphabet = alphabet

    #@with_logging(level=8)
    def _insert(self, seq: list[str], idx: int) -> None:
        insert_idx = random.choice([idx, idx + 1])
        if insert_idx <= len(seq):
            seq.insert(insert_idx, random.choice(self.alphabet))

    #@with_logging(level=8)
    def _replace(self, seq: list[str], idx: int) -> None:
        seq[idx] = random.choice(self.alphabet)

    #@with_logging(level=8)
    def _delete(self, seq: list[str], idx: int) -> None:
        if len(seq) > 1:
            seq.pop(idx)

    #@with_logging(level=8)
    def _swap(self, seq: list[str], idx: int) -> None:
        swap_pos = idx + random.choice([-1, 1])
        if 0 <= swap_pos < len(seq):
            seq[idx], seq[swap_pos] = seq[swap_pos], seq[idx]

In [6]:
import torch
import numpy as np
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, df, preprocessor, masking_percentage):
        """
        Args:
            df (list or array-like): The dataset of sequences.
            preprocessor: An object that can process (tokenize) text -> list of token IDs.
            masking_weights (tuple or list): The probability thresholds for the types of masking.
                Example: (p_mask, p_mask + p_random, 1.0)
                i.e., if random < masking_weights[0], mask token
                      elif random < masking_weights[1], random token
                      else do nothing
            masking_percentage (float): Probability that a non-special token *attempts* to be replaced
                                        by [MASK] or random.
        """
        self.df = df
        self.preprocessor = preprocessor
        self.masking_percentage = masking_percentage
        self.ignore_index = -100

    def __len__(self):
        return len(self.df)

    def _mask(self, input_seq):
        """
        Perform in-place masking (and label creation) for MLM.
        Returns:
            masked_seq: The sequence with some tokens replaced by [MASK].
            labels: An array with original token IDs at masked positions, and -100 at unmasked positions.
        """
        seq = input_seq.copy()
        labels = seq.copy()

        # Identify maskable tokens (excluding special tokens)
        special_tokens = self.preprocessor.vocab.get_special_tokens()
        mask_candidates = np.array([1 if token not in special_tokens else 0 for token in seq])

        # Randomly generate values to decide masking
        probability_array = np.random.rand(len(seq))

        # Apply masking
        for idx in range(len(seq)):
            if mask_candidates[idx] == 0 or probability_array[idx] >= self.masking_percentage:
                labels[idx] = self.ignore_index  # Mark as non-masked
                continue

            seq[idx] = self.preprocessor.vocab.get_id("MASK")

        return seq, labels

    def _create_attention_mask(self, input_seq):
        """
        Returns an attention mask for the input sequence:
         - 1 where token != PAD
         - 0 where token == PAD
        """
        pad_id = self.preprocessor.vocab.get_id("PAD")
        attention_mask = [1 if token != pad_id else 0 for token in input_seq]
        return attention_mask

    def _add_special_tokens(self, seq):
        """
        Adds [CLS] and [SEP] tokens to the sequence.

        Args:
            seq (list): A list of token IDs representing the sequence.

        Returns:
            list: The sequence with [CLS] and [SEP] tokens added.
        """
        cls_token_id = self.preprocessor.vocab.get_id("CLS")
        sep_token_id = self.preprocessor.vocab.get_id("SEP")

        # Add [CLS] at the start and [SEP] at the end
        return [cls_token_id] + seq + [sep_token_id]

    def __getitem__(self, idx):
        """
        Returns:
        input_ids_tensor: The masked input_ids (list of token IDs as a tensor).
        mlm_labels: The MLM labels (list of token IDs or -100 for unmasked positions as a tensor).
        attention_mask_tensor: The attention mask for the sequence as a tensor.
        """
        seq = self.df.iloc[idx]
        # Convert raw item to a list of token IDs
        preprocessed_seq = self.preprocessor.process(seq)

        # Add special tokens to finalize the sequence structure
        finalized_sequence = self._add_special_tokens(preprocessed_seq)

        # Create masked input and the MLM labels
        masked_input_ids, mlm_labels = self._mask(finalized_sequence)

        # Create the attention mask
        attention_mask = self._create_attention_mask(masked_input_ids)

        # Convert to torch tensors
        input_ids_tensor = torch.tensor(masked_input_ids, dtype=torch.long)
        mlm_labels_tensor = torch.tensor(mlm_labels, dtype=torch.long)
        attention_mask_tensor = torch.tensor(attention_mask, dtype=torch.long)

        return {
            "original_seq": seq,
            "input_ids": input_ids_tensor,
            "labels": mlm_labels_tensor,
            "attention_mask": attention_mask_tensor
        }

In [18]:
import torch.nn as nn
import tqdm as notebook_tqdm
from transformers import BertModel, BertConfig

class singleClassHead(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_rate=0.1):
        """
        A wrapper for a predefined nn.Sequential layout:
        - Linear(input_size, hidden_size)
        - ReLU
        - Dropout
        - Linear(hidden_size, output_size)

        Args:
        - input_size (int): Size of the input features.
        - hidden_size (int): Size of the hidden layer.
        - output_size (int): Size of the output features.
        - dropout_rate (float): Dropout probability.
        """
        super(singleClassHead, self).__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.sequential(x)


class Bertax(nn.Module):
    def __init__(self,num_layers=8,
            num_attention_heads=4,
            hidden_size=512,
            intermediate_size=2048,
            vocab_size=69,
            max_position_embeddings=22,
            num_classes=10,
            dropout_rate=0.1):

        super(Bertax, self).__init__()

        # Initialized to pretrain mode
        self.mode = "pretrain"

        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=hidden_size,
            num_hidden_layers=num_layers,
            num_attention_heads=num_attention_heads,
            intermediate_size=intermediate_size,
            max_position_embeddings=max_position_embeddings,
            hidden_dropout_prob=dropout_rate,
            attention_probs_dropout_prob=dropout_rate
        )

        self.bert = BertModel(config)

        self.mlm_head = nn.Linear(hidden_size, vocab_size)

        #TODO: I want to make a wrapper class for the nn.sequential object, so I can factor it out from this model. Help me write that calss. 
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2 ),
            nn.ReLU(),
            nn.Dropout(p = dropout_rate),
            nn.Linear(hidden_size // 2, num_classes)
        )

    def preTrainMode(self):
        self.mode = "pretrain"

    def classifyMode(self):
        self.mode = "classify"

    def forward(self, input_ids, attention_mask = None):
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask
            )
        sequence_output = outputs.last_hidden_state
        pooled_output = outputs.pooler_output

        if self.mode == "pretrain":
            #USE MLM-head for pre-training
            return self.mlm_head(sequence_output)
        if self.mode == "classify":
                return self.classification_head(pooled_output)
        else:
             raise ValueError(f"Invalid mode: {self.mode}. Use 'pretrain' or 'classify'.")




In [34]:
class ModularBertax(nn.Module):
    def __init__(self,encoder, mlm_head, classification_head):

        super(ModularBertax, self).__init__()

        # Initialized to pretrain mode
        self.mode = "pretrain"
        
        self.bert = encoder
        self.mlm_head = mlm_head
        self.classification_head = classification_head

    def preTrainMode(self):
        self.mode = "pretrain"

    def classifyMode(self):
        self.mode = "classify"

    def forward(self, input_ids, attention_mask = None):
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask
            )
        sequence_output = outputs.last_hidden_state
        pooled_output = outputs.pooler_output

        if self.mode == "pretrain":
            #USE MLM-head for pre-training
            return self.mlm_head(sequence_output)
        if self.mode == "classify":
                return self.classification_head(pooled_output)
        else:
             raise ValueError(f"Invalid mode: {self.mode}. Use 'pretrain' or 'classify'.")

In [40]:
class singleClassHead(nn.Module):
    def __init__(self, in_features, hidden_layer_size, out_features, dropout_rate=0.1):
        """
        A wrapper for a predefined nn.Sequential layout:
        - Linear(input_size, hidden_size)
        - ReLU
        - Dropout
        - Linear(hidden_size, output_size)

        Args:
        - input_size (int): Size of the input features.
        - hidden_size (int): Size of the hidden layer.
        - output_size (int): Size of the output features.
        - dropout_rate (float): Dropout probability.
        """

        self.in_features = in_features
        self.hidden_layer_size = hidden_layer_size
        self.out_features = out_features
        self.dropout_rate = dropout_rate

        super(singleClassHead, self).__init__()
        
        self.sequential = nn.Sequential(
            nn.Linear(self.in_features, self.hidden_layer_size),
            nn.ReLU(),
            nn.Dropout(p=self.dropout_rate),
            nn.Linear(self.hidden_layer_size, self.out_features)
        )

    def forward(self, x):
        return self.sequential(x)
    
class MLMHead(nn.Module):
    def __init__(self, in_features, hidden_layer_size, out_features, dropout_rate=0.1):
        """
        A wrapper for a predefined nn.Sequential layout:
        - Linear(input_size, hidden_size)
        - ReLU
        - Dropout
        - Linear(hidden_size, output_size)

        Args:
        - input_size (int): Size of the input features.
        - hidden_size (int): Size of the hidden layer.
        - output_size (int): Size of the output features.
        - dropout_rate (float): Dropout probability.
        """

        self.in_features = in_features
        self.hidden_layer_size = hidden_layer_size
        self.out_features = out_features
        self.dropout_rate = dropout_rate

        super(MLMHead, self).__init__()
        
        self.sequential = nn.Sequential(
            nn.Linear(self.in_features, self.hidden_layer_size),
            nn.ReLU(),
            nn.Dropout(p=self.dropout_rate),
            nn.Linear(self.hidden_layer_size, self.out_features)
        )

    def forward(self, x):
        return self.sequential(x)

In [41]:

## CONFIG ------------------------------------------
#Params for encoder:
max_position_embeddings = 202 #max input dimension

num_hidden_layers = 10
num_attention_heads = 4 
hidden_size = 256  # hidden_size mod num_attention_heads = 0
intermediate_size = 4 * hidden_size  # intermediate_size = 4 x hiddensize (this is the standard transformers design) NOT num_attentionheads, but just 4

mlm_dropout_rate = 0.1
hidden_dropout_prob = 0.1
attention_probs_dropout_prob = 0.1

#params for classification head:
input_size = hidden_size
hidden_layer_size = input_size/2
## CONFIG ------------------------------------------

vocab_size = 69# vocab_size #dynamic
output_size = 10# num_classes #dynamic 

config = BertConfig(
        vocab_size=vocab_size,
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        num_attention_heads=num_attention_heads,
        intermediate_size=intermediate_size,
        max_position_embeddings=max_position_embeddings,
        hidden_dropout_prob=hidden_dropout_prob,
        attention_probs_dropout_prob=attention_probs_dropout_prob
    )
encoder = BertModel(config)

mlm_head = MLMHead(
    in_features = hidden_size,
    hidden_layer_size = hidden_size //2,
    out_features = vocab_size,
    dropout_rate = mlm_dropout_rate
)


classification_head = singleClassHead(
    in_features = hidden_size,
    hidden_layer_size = hidden_size//2,
    out_features = vocab_size,
    dropout_rate = mlm_dropout_rate
    )

model = ModularBertax(
    encoder = encoder,
    mlm_head=mlm_head,
    classification_head=classification_head
)

In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datetime import datetime
import json
from tqdm import tqdm

class MLMtrainer:
    def __init__(self,
            model: nn.Module,
            train_loader: DataLoader,
            val_loader: DataLoader,
            weight_save_path: str = "best_pre_train_weights.pt"):


        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        print(f"Training on device {self.device} and it is awesome!!!")

        self.train_loader = train_loader
        self.val_loader = val_loader

        self.criteronMLM = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(self.model.parameters(), lr = 5e-5) # TODO: How to (and where) to introduce a lr-scheduler?
        self.best_val_loss = float('inf')

    def _run_epoch(self, epoch_nr):

        self.model.train()
        total_loss, mlm_correct, total_mlm = 0, 0, 0
        progress_bar = tqdm(self.train_loader, desc=f"Training Epoch {epoch_nr + 1}", leave=False)

        ignore_index = self.train_loader.dataset.ignore_index

        for batch in progress_bar:
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)  # Fixed misplaced parentheses
            mlm_labels = batch["labels"].to(self.device)

            self.optimizer.zero_grad()
            mlm_logits = self.model(input_ids, attention_mask)

            # Compute MLM loss
            loss = self.criteronMLM(
                mlm_logits.view(-1, self.model.mlm_head.out_features),
                mlm_labels.view(-1)
            )

            loss.backward()
            self.optimizer.step()

            # Accumulate total loss
            total_loss += loss.item()

            # Compute MLM accuracy
            mlm_preds = mlm_logits.argmax(dim=-1)
            mlm_correct += (mlm_preds == mlm_labels).masked_select(mlm_labels != ignore_index).sum().item()
            total_mlm += (mlm_labels != ignore_index).sum().item()

        # Calculate average loss and MLM accuracy
        avg_loss = total_loss / len(self.train_loader)
        mlm_acc = mlm_correct / total_mlm if total_mlm > 0 else 0

        return avg_loss, mlm_acc

    def _validate_epoch(self, epoch_nr):
        self.model.eval()
        total_loss, mlm_correct, total_mlm = 0, 0, 0
        progress_bar = tqdm(self.val_loader, desc=f"Validation Epoch {epoch_nr + 1}", leave=False)

        ignore_index = self.val_loader.dataset.ignore_index
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            mlm_labels = batch["labels"].to(self.device)

            with torch.no_grad():
                mlm_logits = self.model(input_ids, attention_mask)

                # Compute MLM loss
                loss = self.criteronMLM(
                    mlm_logits.view(-1, self.model.mlm_head.out_features),
                    mlm_labels.view(-1)
                )

                # Accumulate total loss
                total_loss += loss.item()

                # Compute MLM accuracy
                mlm_preds = mlm_logits.argmax(dim=-1)
                mlm_correct += (mlm_preds == mlm_labels).masked_select(mlm_labels != ignore_index).sum().item()
                total_mlm += (mlm_labels != ignore_index).sum().item()

        # Calculate average loss and MLM accuracy
        avg_loss = total_loss / len(self.val_loader)
        mlm_acc = mlm_correct / total_mlm if total_mlm > 0 else 0

        return avg_loss, mlm_acc


    def train(self, num_epochs = 10):
        for epoch_nr in range(num_epochs):
            # Run training for one epoch
            train_avg_loss, train_mlm_acc = self._run_epoch(epoch_nr)
            val_avg_loss, val_mlm_acc = self._validate_epoch(epoch_nr)

            if val_avg_loss < self.best_val_loss:
                self.best_val_loss = val_avg_loss
                torch.save(self.model.state_dict(), "best_MLM_weights.pt")

            # Report training metrics
            print(f"Epoch {epoch_nr + 1}/{num_epochs}")
            print(f"Train Avg Loss: {train_avg_loss:.4f}, Train MLM Accuracy: {train_mlm_acc:.4f}")
            print(f"Val Avg Loss: {val_avg_loss:.4f}, Val MLM Accuracy: {val_mlm_acc:.4f}")



# Pretraining the model

In [11]:
import torch
from preprocessing.preprocessor import Preprocessor
from preprocessing.augmentation import IdentityStrategy, BaseStrategy
from preprocessing.tokenization import KmerStrategy
from preprocessing.padding import PEndStrategy
from preprocessing.truncation import TEndStrategy

from vocab import Vocabulary, KmerVocabConstructor
from utils.dataset import fasta2pandas
from utils.dataset import filter_taxonomy

from sklearn.model_selection import train_test_split

import json

#Config: -------------------------------------------------------------------------------------------
FILE_PATH = 'data/raw.fasta'
SAVE_PATH = "pretrained_model.pt"

n_test = 10

modification_probability: float = 0.05
alphabet = ["A", "C", "G", "T"]
k = 3
optimal_length = 200 #TODO: Why, when I change this to say 200, does the last line in this cell break?


# Training config: ------------------------------------------------------------------------------
num_epochs = 1

masking_percentage = 0.05

batch_size = 128

small_set = True

#Model config: ------------------------------------------------------------------------------
num_layers = 10
num_attention_heads = 4
hidden_size = 256
intermediate_size = 1024
dropout_rate = 0.05

# Set up vocabulary ------------------------------------------------------------------------------

constructor = KmerVocabConstructor(k=k, alphabet=alphabet)
vocab = Vocabulary()
vocab.build_from_constructor(constructor, data=[])

vocab_path = "vocab.json"
vocab.save(vocab_path)

# Set up preprocessor-------------------------------------------------------------------------------
# Training preprocessor
sequence_modifier = SequenceModifier(alphabet=alphabet)

augmentation_strategy_train = BaseStrategy(
    modifier=sequence_modifier,
    alphabet=alphabet,
    modification_probability=modification_probability
)

augmentation_strategy_val = IdentityStrategy(
    modifier=sequence_modifier,
    alphabet=alphabet,
    modification_probability = 0 # No augmentation for validation
)

tokenization_strategy = KmerStrategy(
    k=k,
    padding_alphabet=alphabet
)

padding_strategy = PEndStrategy(
    optimal_length=optimal_length
)

truncation_strategy = TEndStrategy(
    optimal_length=optimal_length
)

preprocessor_train = Preprocessor(
    augmentation_strategy=augmentation_strategy_train,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

# The only thing that separates the preprocessors is the augmentation strategies.
preprocessor_val = Preprocessor(
    augmentation_strategy=augmentation_strategy_val,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

# Set up pre-training data ------------------------------------------------------------------------
df = fasta2pandas(FILE_PATH)

if small_set: #TODO: Better
    df = df[:n_test]

all_sequences = df["sequence"]


# Filtered sequences for the fine-tuning
filtered_sequences = ...

# Split into train and validation set for pre-training
pretrain_sequences, preval_sequences = train_test_split(
    all_sequences,
    test_size=0.1,
    random_state=42
    )

print(f"Pre-training sequences: {len(pretrain_sequences)}")
print(f"Validation sequences: {len(preval_sequences)}")


train_dataset = CustomDataset(
    pretrain_sequences,
    preprocessor_train,
    masking_percentage
    )

val_dataset = CustomDataset(
    preval_sequences,
    preprocessor_val,
    masking_percentage
    )

train_loader = DataLoader(
    dataset = train_dataset,
    batch_size = batch_size,
    shuffle = True
    )

val_loader = DataLoader(
    dataset = val_dataset,
    batch_size = batch_size,
    shuffle = True
    )

my_model = Bertax(
    num_layers = num_layers,
    num_attention_heads= num_attention_heads,
    hidden_size = hidden_size,
    intermediate_size =intermediate_size,
    vocab_size = len(vocab),
    max_position_embeddings = optimal_length + 2,
    dropout_rate= dropout_rate,
    num_classes = 19 # 19 classes at phylum level with no uncertainties
    )

mlm_trainer = MLMtrainer(
    my_model,
    train_loader = train_loader,
    val_loader = val_loader
    )



mlm_trainer.train(num_epochs)
torch.save(my_model.state_dict(), SAVE_PATH)
print(f"Model's state dictionary saved to {SAVE_PATH}")

Pre-training sequences: 9
Validation sequences: 1
Training on device cpu and it is awesome!!!


Epoch 1/1
Train Avg Loss: 4.2924, Train MLM Accuracy: 0.0267
Val Avg Loss: 4.7520, Val MLM Accuracy: 0.0000
Model's state dictionary saved to pretrained_model.pt


# Fine Tuning the model

In [44]:
class LabelEncoder:
    def __init__(self, labels):
        # Extract unique labels from the provided array
        unique_labels = set(labels)

        # Create dictionaries for encoding and decoding
        self.label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
        self.index_to_label = {idx: label for idx, label in enumerate(unique_labels)}

    def encode(self, label):
        # Encode label to its corresponding index
        return self.label_to_index.get(label, None)

    def decode(self, index):
        # Decode index to its corresponding label
        return self.index_to_label.get(index, None)


In [47]:
class ClassificationDataset(Dataset):
    def __init__(self,
            df,
            preprocessor,
            label_encoder,
            target_column = "species"):

        self.df = df
        self.preprocessor = preprocessor
        self.label_encoder = label_encoder
        self.target_column = target_column

    def __len__(self):
        return len(self.df)

    def _create_attention_mask(self, input_seq):
        """
        Returns an attention mask for the input sequence:
         - 1 where token != PAD
         - 0 where token == PAD
        """
        pad_id = self.preprocessor.vocab.get_id("PAD")
        attention_mask = [1 if token != pad_id else 0 for token in input_seq]
        return attention_mask

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        sequence = row["sequence"]
        preprocessed_sequence = self.preprocessor.process(sequence)
        attention_mask = self._create_attention_mask(preprocessed_sequence)

        label = row[self.target_column]

        return {
            "original_seq": sequence,
            "input_ids": torch.tensor(preprocessed_sequence, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "label": self.label_encoder.encode(label),
            "encoded_label": self.label_encoder.encode(label)
        }


In [48]:
class ClassificationTrainer:
    def __init__(self,
            model: nn.Module,
            train_loader: DataLoader,
            val_loader: DataLoader,
            weight_save_path: str = "best_classification_weights.pt"):

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        print(f"Training on device {self.device} and it is awesome!!!")

        self.train_loader = train_loader
        self.val_loader = val_loader

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(self.model.parameters(), lr=5e-6)
        self.best_val_loss = float('inf')

    def _run_epoch(self, epoch_nr):
        self.model.train()
        total_loss, correct, total = 0, 0, 0
        progress_bar = tqdm(self.train_loader, desc=f"Training Epoch {epoch_nr + 1}", leave=False)

        for batch in progress_bar:
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            labels = batch["encoded_label"].to(self.device)

            self.optimizer.zero_grad()
            logits = self.model(input_ids, attention_mask)

            loss = self.criterion(logits, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += len(labels)

    def _validate_epoch(self, epoch_nr):
        self.model.eval()
        total_loss, correct, total = 0, 0, 0
        progress_bar = tqdm(self.val_loader, desc=f"Validation Epoch {epoch_nr + 1}", leave=False)

        for batch in progress_bar:
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            labels = batch["encoded_label"].to(self.device)

            with torch.no_grad():
                logits = self.model(input_ids, attention_mask)
                loss = self.criterion(logits, labels)

                total_loss += loss.item()
                correct += (logits.argmax(dim=-1) == labels).sum().item()
                total += len(labels)

        return total_loss / len(self.val_loader), correct / total

    def train(self, num_epochs=10):
        for epoch_nr in range(num_epochs):
            self._run_epoch(epoch_nr)
            val_loss, val_acc = self._validate_epoch(epoch_nr)

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), "best_classification_weights.pt")

            print(f"Epoch {epoch_nr + 1}/{num_epochs}")
            print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")


In [49]:
# Just to check how many labels there are
df = fasta2pandas(FILE_PATH)
#print(f"Number of sequences: {len(df)}")
filtered_df = filter_taxonomy(df, startAt='phylum', endAt='phylum', phylumCertainty=True)
print(f"Number of sequences after filtering: {len(filtered_df)}")
all_labels = filtered_df["phylum"]
print(f"Number of unique labels: {len(set(all_labels))}")

Applying filters...
Filtering complete.

Handling DNA ambiguity codes...
Processing row 0...
Processing row 10000...
Processing row 20000...
Processing row 30000...
Processing row 40000...
Processing row 50000...
Processing row 60000...
Processing row 70000...
Processing row 80000...
Processing row 90000...
Processing complete.
Number of sequences after filtering: 86490
Number of unique labels: 19


In [50]:
# Fine tuning
df = fasta2pandas(FILE_PATH)
filtered_df = filter_taxonomy(df, startAt='phylum', endAt='phylum', phylumCertainty=True)

if small_set: #TODO: Better
    filtered_df = filtered_df[:n_test]

#all_sequences = filtered_df["sequence"]
all_labels = filtered_df["phylum"]
print(f"Number of unique labels: {len(set(all_labels))}")

label_encoder = LabelEncoder(all_labels)

#print(f"Number of unique labels: {len(label_encoder.label_to_index)}")

# Split into train and validation set for pre-training
train_df, val_df = train_test_split(
    filtered_df,
    test_size=0.1,
    random_state=42
    )

#data sets
train_dataset = ClassificationDataset(
    df=train_df,
    preprocessor=preprocessor_train,
    label_encoder=label_encoder,
    target_column="phylum"
    )

val_dataset = ClassificationDataset(
    df=val_df,
    preprocessor=preprocessor_val,
    label_encoder=label_encoder,
    target_column="phylum"
    )


#data loaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
    )

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=True
    )


my_model.classifyMode()

classification_trainer = ClassificationTrainer(
    my_model,
    train_loader=train_loader,
    val_loader=val_loader
    )

classification_trainer.train(num_epochs)





Applying filters...
Filtering complete.

Handling DNA ambiguity codes...
Processing row 0...
Processing row 10000...
Processing row 20000...
Processing row 30000...
Processing row 40000...
Processing row 50000...
Processing row 60000...
Processing row 70000...
Processing row 80000...
Processing row 90000...
Processing complete.
Number of unique labels: 4
Length of train dataset: 9
Length of val dataset: 1
Training on device cpu and it is awesome!!!


Epoch 1/1
Val Loss: 3.0221, Val Accuracy: 0.0000


In [51]:
import torch
from preprocessing.preprocessor import Preprocessor
from preprocessing.augmentation import IdentityStrategy, BaseStrategy
from preprocessing.tokenization import KmerStrategy
from preprocessing.padding import PEndStrategy
from preprocessing.truncation import TEndStrategy
from vocab import Vocabulary, KmerVocabConstructor
from utils.dataset import fasta2pandas
from sklearn.model_selection import train_test_split

# Model and training configuration
CONFIG = {
    "FILE_PATH": "data/raw.fasta",
    "SAVE_PATH": "pretrained_model.pt",
    "n_test": 10,
    "modification_probability": 0.05,
    "alphabet": ["A", "C", "G", "T"],
    "k": 3,
    "optimal_length": 200,

    # Training parameters
    "num_epochs": 1,
    "masking_percentage": 0.05,
    "batch_size": 128,
    "small_set": True,

    # Model configuration
    "num_layers": 10,
    "num_attention_heads": 4,
    "hidden_size": 256,
    "intermediate_size": 1024,  # 4 * hidden_size
    "dropout_rate": 0.05,
    "num_classes": 19,
    "mlm_dropout_rate": 0.1
}

# Set up vocabulary
constructor = KmerVocabConstructor(k=CONFIG["k"], alphabet=CONFIG["alphabet"])
vocab = Vocabulary()
vocab.build_from_constructor(constructor, data=[])
vocab_path = "vocab.json"
vocab.save(vocab_path)

# Set up preprocessors
sequence_modifier = SequenceModifier(alphabet=CONFIG["alphabet"])
augmentation_strategy_train = BaseStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=CONFIG["modification_probability"]
)

augmentation_strategy_val = IdentityStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=0
)

tokenization_strategy = KmerStrategy(
    k=CONFIG["k"],
    padding_alphabet=CONFIG["alphabet"]
)

padding_strategy = PEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

truncation_strategy = TEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

preprocessor_train = Preprocessor(
    augmentation_strategy=augmentation_strategy_train,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

preprocessor_val = Preprocessor(
    augmentation_strategy=augmentation_strategy_val,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

# Set up pre-training data
df = fasta2pandas(CONFIG["FILE_PATH"])
if CONFIG["small_set"]:
    df = df[:CONFIG["n_test"]]
all_sequences = df["sequence"]

pretrain_sequences, preval_sequences = train_test_split(
    all_sequences, test_size=0.1, random_state=42
)

print(f"Pre-training sequences: {len(pretrain_sequences)}")
print(f"Validation sequences: {len(preval_sequences)}")

train_dataset = CustomDataset(
    pretrain_sequences, preprocessor_train, CONFIG["masking_percentage"]
)

val_dataset = CustomDataset(
    preval_sequences, preprocessor_val, CONFIG["masking_percentage"]
)

train_loader = DataLoader(
    dataset=train_dataset, batch_size=CONFIG["batch_size"], shuffle=True
)

val_loader = DataLoader(
    dataset=val_dataset, batch_size=CONFIG["batch_size"], shuffle=True
)

# Set up the model encoder and heads
bert_config = BertConfig(
    vocab_size=len(vocab),
    hidden_size=CONFIG["hidden_size"],
    num_hidden_layers=CONFIG["num_layers"],
    num_attention_heads=CONFIG["num_attention_heads"],
    intermediate_size=CONFIG["intermediate_size"],
    max_position_embeddings=CONFIG["optimal_length"] + 2,
    hidden_dropout_prob=CONFIG["dropout_rate"],
    attention_probs_dropout_prob=CONFIG["dropout_rate"]
)

encoder = BertModel(bert_config)

mlm_head = MLMHead(
    in_features=CONFIG["hidden_size"],
    hidden_layer_size=CONFIG["hidden_size"] // 2,
    out_features=len(vocab),
    dropout_rate=CONFIG["mlm_dropout_rate"]
)

classification_head = MLMHead(
    in_features=CONFIG["hidden_size"],
    hidden_layer_size=CONFIG["hidden_size"] // 2,
    out_features=CONFIG["num_classes"],
    dropout_rate=CONFIG["dropout_rate"]
)

model = ModularBertax(
    encoder=encoder,
    mlm_head=mlm_head,
    classification_head=classification_head
)

# Train the model
mlm_trainer = MLMtrainer(
    model=model, train_loader=train_loader, val_loader=val_loader
)

mlm_trainer.train(CONFIG["num_epochs"])
torch.save(model.state_dict(), CONFIG["SAVE_PATH"])
print(f"Model's state dictionary saved to {CONFIG['SAVE_PATH']}")


Pre-training sequences: 9
Validation sequences: 1
Training on device cpu and it is awesome!!!


Epoch 1/1
Train Avg Loss: 4.2654, Train MLM Accuracy: 0.0108
Val Avg Loss: 4.3449, Val MLM Accuracy: 0.0000
Model's state dictionary saved to pretrained_model.pt


In [1]:
# Fine tuning
df = fasta2pandas(FILE_PATH)
filtered_df = filter_taxonomy(df, startAt='phylum', endAt='phylum', phylumCertainty=True)

if small_set: #TODO: Better
    filtered_df = filtered_df[:n_test]

#all_sequences = filtered_df["sequence"]
all_labels = filtered_df["phylum"]
print(f"Number of unique labels: {len(set(all_labels))}")

label_encoder = LabelEncoder(all_labels)

#print(f"Number of unique labels: {len(label_encoder.label_to_index)}")

# Split into train and validation set for pre-training
train_df, val_df = train_test_split(
    filtered_df,
    test_size=0.1,
    random_state=42
    )

#data sets
train_dataset = ClassificationDataset(
    df=train_df,
    preprocessor=preprocessor_train,
    label_encoder=label_encoder,
    target_column="phylum"
    )

val_dataset = ClassificationDataset(
    df=val_df,
    preprocessor=preprocessor_val,
    label_encoder=label_encoder,
    target_column="phylum"
    )


#data loaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
    )

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=True
    )


model.classifyMode()

classification_trainer = ClassificationTrainer(
    model,
    train_loader=train_loader,
    val_loader=val_loader
    )

classification_trainer.train(num_epochs)



NameError: name 'fasta2pandas' is not defined